# Embedding of evoked responses and TCA
Inspired by 'ripples' paper where time are used as features to investigate diversity of responses across age and cells

In [ ]:
import numpy as np
import os

session_paths = [
    'data_proc_ssd/jm/jm064/2025-11-13_s',
    'data_proc_ssd/jm/jm064/2025-11-14_s',
    'data_proc_ssd/jm/jm064/2025-11-15_s',
    'data_proc_ssd/jm/jm064/2025-11-16_s',
    'data_proc_ssd/jm/jm064/2025-11-17_s',
    'data_proc_ssd/jm/jm064/2025-11-18_s'
]

t2p_indices_path = 'data_proc_ssd/jm/jm064/track2p/plane0_suite2p_indices.npy'

st_analysis = False


all_resp_evoked = []
for session_path in session_paths:
    all_resp_evoked.append(np.load(os.path.join(session_path, 'resp_evoked', 'resp_evoked.npy'), allow_pickle=True).item())

n_features = all_resp_evoked[0]['resp_mean'].shape[1]
n_days = len(all_resp_evoked)
n_neurons = all_resp_evoked[0]['resp_mean'].shape[0]

In [ ]:
n_trials = 60
all_resp_mean = np.zeros((n_days, n_neurons, n_features))
all_resp = np.zeros((n_days, n_trials, n_neurons, n_features))


for i, resp_evoked in enumerate(all_resp_evoked):
    all_resp_mean[i] = resp_evoked['resp_mean']
    print(resp_evoked['resp'].shape)
    all_resp[i] = resp_evoked['resp']

# TODO: THE MATCHING DOESN'T MAKE SENSE...

# now plot some example neurons across days
import matplotlib.pyplot as plt
neuron_idxs = np.int64(np.linspace(0, n_neurons-1, 50))

for neuron_idx in neuron_idxs:
    fig, axs = plt.subplots(1, n_days, figsize=(15, 1))
    for day in range(n_days):
        axs[day].plot(all_resp_mean[day, neuron_idx])
        axs[day].set_title(f'Day {day+1}')
    plt.suptitle(f'Neuron {neuron_idx}')
    plt.show()

In [ ]:
print(all_resp.shape)
# now flatten the data for dimensionality reduction (neurons*trials*days x features)
all_resp_flat = all_resp.reshape(-1, n_features)
print(all_resp_flat.shape)
# now label rows by day


In [ ]:
# now import and run umap
import umap
from sklearn.decomposition import PCA

In [ ]:
def zscore_rows(X):
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

In [ ]:
# flatten all_resp_mean along the days dimension and add labels (to color code the points by day)
data_mn = all_resp_mean.reshape(-1, n_features)
data_st = all_resp_flat
# # Compute the PSD: Instead of feeding raw time series into UMAP, feed it the Power Spectral Density (PSD) or the magnitude of the Fast Fourier Transform (FFT).
# data_psd = np.abs(np.fft.fft(data, axis=1))

# data = zscore_rows(data_psd)

labels_mn = np.repeat(np.arange(n_days), n_neurons)
labels_st = np.repeat(np.arange(n_days), n_neurons*n_trials)


In [ ]:
# now fit umap and visualise
reducer_mn = umap.UMAP(
    random_state=42,
    n_neighbors=15
)
emb_umap_mn = reducer_mn.fit_transform(data_mn)

In [ ]:
if st_analysis:
    reducer_st = umap.UMAP(
        random_state=42,
        n_neighbors=15
    )

    emb_umap_st = reducer_st.fit_transform(data_st)
else:
    print('ST analysis is disabled, skipping ST UMAP embedding.')

In [ ]:
nrn_idx = 306
nrn_idx_days = [nrn_idx + i*n_neurons for i in range(n_days)]

In [ ]:
plt.figure(figsize=(14, 10), dpi=300)
plt.scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], s=30)
plt.axis('off')
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Postnatal day', fontsize=24)

In [ ]:
plt.figure(figsize=(14, 10), dpi=300)
plt.scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], c=labels_mn+8, s=30, cmap='plasma')
plt.axis('off')
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Postnatal day', fontsize=24)

In [ ]:
if st_analysis:
    plt.figure(figsize=(14, 10), dpi=300)
    plt.scatter(emb_umap_st[:, 0], emb_umap_st[:, 1], c=labels_st+8, s=0.1, cmap='viridis', alpha=0.5)
    plt.axis('off')
    cbar = plt.colorbar()
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label('Postnatal day', fontsize=24)
else:
    print('ST analysis is disabled, skipping ST UMAP plot.')

In [ ]:
# now get the centroid of the points corresponding to all of the trials of a given neuron on a given day

if st_analysis:
    nrn_idx = 33
    all_centroid = np.zeros((n_days, 2))
    for d in range(n_days):
        # get indices of the points corresponding to the trials of neuron nrn_idx on day d
        trial_indices = np.where(labels_st == d)[0]
        neuron_trial_indices = trial_indices[trial_indices % n_neurons == nrn_idx]
        neuron_trial_points = emb_umap_st[neuron_trial_indices]
        all_centroid[d] = neuron_trial_points.mean(axis=0)

        print(f'Centroid for neuron {nrn_idx} on day {d+1}: {all_centroid[d]}')
else:
    print('ST analysis is disabled, skipping centroid calculation.')

In [ ]:
ex_nrn_idxs = [33, 94, 284, 60, 47, 127, 282, 329]

In [ ]:
if st_analysis:
    for nrn_idx in ex_nrn_idxs:
        
        all_centroid = np.zeros((n_days, 2))
        for d in range(n_days):
            # get indices of the points corresponding to the trials of neuron nrn_idx on day d
            trial_indices = np.where(labels_st == d)[0]
            neuron_trial_indices = trial_indices[trial_indices % n_neurons == nrn_idx]
            neuron_trial_points = emb_umap_st[neuron_trial_indices]
            all_centroid[d] = neuron_trial_points.mean(axis=0)

            print(f'Centroid for neuron {nrn_idx} on day {d+1}: {all_centroid[d]}')

        plt.figure(figsize=(14, 10), dpi=300)
        plt.scatter(emb_umap_st[:, 0], emb_umap_st[:, 1], c=labels_st+8, s=0.1, cmap='viridis', alpha=0.5)
        # now plot the centroids
        plt.scatter(all_centroid[:, 0], all_centroid[:, 1], c=np.arange(n_days)+8, s=50, cmap='viridis', label=f'Neuron {nrn_idx} centroids', zorder=3)
        plt.plot(all_centroid[:, 0], all_centroid[:, 1], c='grey', linewidth=3, zorder=2, label='Trajectory')
        plt.legend()
        plt.axis('off')
        cbar = plt.colorbar()
        cbar.ax.tick_params(labelsize=18)
        cbar.set_label('Postnatal day', fontsize=24)
        plt.show()
else:
    print('ST analysis is disabled, skipping centroid plotting.')

In [ ]:
for nrn_idx in ex_nrn_idxs:
    nrn_idx_days = [nrn_idx + i*n_neurons for i in range(n_days)]
    
    fig, axs = plt.subplot_mosaic(mosaic='AAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nAAAAAA\nBCDEFG', figsize=(10, 10), dpi=300)
    # make the BCDEFG share y axis
    for ax in ['B', 'C', 'D', 'E', 'F', 'G']:
        axs[ax].sharey(axs['B'])
    axs['A'].scatter(emb_umap_mn[:, 0], emb_umap_mn[:, 1], c=labels_mn, s=5, cmap='plasma',alpha=0.7, zorder=0)
    axs['A'].scatter(emb_umap_mn[nrn_idx_days, 0], emb_umap_mn[nrn_idx_days, 1], c=np.arange(n_days), s=200, cmap='plasma')
    axs['A'].plot(emb_umap_mn[nrn_idx_days, 0], emb_umap_mn[nrn_idx_days, 1], c='gray', alpha=0.5, label=f'Trajectory of neuron {nrn_idx}', linewidth=5)
    axs['A'].set_title('UMAP embedding of evoked responses')
    axs['A'].set_xlabel('UMAP 1')
    axs['A'].set_ylabel('UMAP 2')
    # remove axis
    axs['A'].axis('off')
    # add legend to top left corner of the plot
    axs['A'].legend(loc='upper left', frameon=False)
    # add colormap labelled with the day numbers
    other_days = 'BCDEFG'
    for day in range(n_days):
        # get color based on 'plasma' colormap and the day index
        color = plt.cm.plasma(day/ (n_days-1))
        axs[other_days[day]].plot(all_resp_mean[day, nrn_idx], color=color, linewidth=3.5)
        axs[other_days[day]].set_xticks([])
        axs[other_days[day]].set_yticks([]) 
        axs[other_days[day]].set_axis_off()
        # add f'P{8+day}' to top left corner of the subplot
        axs[other_days[day]].text(0.05, 0.95, f'P{8+day}', transform=axs[other_days[day]].transAxes, fontsize=16, verticalalignment='top', color=color)


In [ ]:
import tensortools as tt

data = all_resp_mean # ... specify a numpy array holding the tensor you wish to fit
# data = all_resp.reshape(n_days, n_trials*n_neurons, n_features) # ... specify a numpy array holding the tensor you wish to fit

# Fit an ensemble of models, 4 random replicates / optimization runs per model rank
ensemble = tt.Ensemble(fit_method="ncp_hals")
ensemble.fit(data, ranks=range(1, 10), replicates=5)

fig, axes = plt.subplots(1, 2)
tt.plot_objective(ensemble, ax=axes[0])   # plot reconstruction error as a function of num components.
tt.plot_similarity(ensemble, ax=axes[1])  # plot model similarity as a function of num components.
fig.tight_layout()

# Plot the low-d factors for an example model, e.g. rank-2, first optimization run / replicate.
num_components = 4
replicate = 0
# color lines in 'C1' 
tt.plot_factors(ensemble.factors(num_components)[replicate], line_kw=[{'color': 'C2'}, {'color': 'C0'}, {'color': 'C3'}])  # plot the low-d factors

plt.show()

In [ ]:
day_components = np.zeros((num_components, n_days))
nrn_components = np.zeros((num_components, n_neurons))
feature_components = np.zeros((num_components, n_features))

for i in range(num_components):
    day_components[i] = ensemble.factors(num_components)[replicate][0][:, i]
    nrn_components[i] = ensemble.factors(num_components)[replicate][1][:, i]
    feature_components[i] = ensemble.factors(num_components)[replicate][2][:, i]

In [ ]:
fig, axs = plt.subplots(num_components, 1, figsize=(2, 5))
for i in range(num_components):
    axs[i].hist(nrn_components[i,:], bins=26)
    axs[i].axis('off')
plt.show()

In [ ]:
import matplotlib.colors as colors


In [ ]:
# now scatter the umap embedding color coded by the TCA component embedding
for i in range(num_components):
    # outer product of neuron and day componentsa
    c = np.outer(day_components[i], nrn_components[i, :]).flatten()
    # now do the log to get a logarithmic color scale (since the values are mostly close to zero, with some large outliers)
    c = np.log(np.abs(c) + 1e-5)  # add a small value to avoid log(0)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(
        emb_umap_mn[:, 0],
        emb_umap_mn[:, 1],
        c=c,
        s=5,
        cmap='viridis'
    )
    plt.axis('off')
    cbar = plt.colorbar(ticks=[])
    cbar.set_label(fr'$TC{i+1}_{{\mathrm{{day}}}}\otimes TC{i+1}_{{\mathrm{{neuron}}}}$ (log scale)', fontsize=12)
    plt.title(f'UMAP embedding colored by TCA component {i+1}')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.show()

In [ ]:
# now scatter the umap embedding color coded by the TCA component embedding
for i in range(num_components):
    # outer product of neuron and day componentsa
    c = np.outer(day_components[i], nrn_components[i, :]).flatten()
    # now do the log to get a logarithmic color scale (since the values are mostly close to zero, with some large outliers)
    c = np.log(np.abs(c) + 1e-5)  # add a small value to avoid log(0)
    
    fig, axs = plt.subplot_mosaic(mosaic='AAA\nAAA\nAAA\nAAA\nAAA\nAAA\nBCD', figsize=(10, 8), dpi=300)
    axs['A'].scatter(
        emb_umap_mn[:, 0],
        emb_umap_mn[:, 1],
        c=c,
        s=5,
        cmap='viridis'
    )
    axs['A'].axis('off')
    cbar = fig.colorbar(axs['A'].collections[0], ax=axs['A'], ticks=[], shrink=0.8)
    cbar.set_label(fr'$TC{i+1}_{{\mathrm{{day}}}}\otimes TC{i+1}_{{\mathrm{{neuron}}}}$ (log scale)', fontsize=12)
    axs['A'].set_title(f'UMAP embedding colored by TCA component {i+1}')
    axs['B'].plot(day_components[i], color='C2')
    axs['B'].set_title(f'Day component {i+1}')
    axs['B'].set_axis_off()
    axs['C'].plot(nrn_components[i, :], color='C0')
    axs['C'].set_title(f'Neuron component {i+1}')
    axs['C'].set_axis_off()
    axs['D'].plot(feature_components[i, :], color='C3')
    axs['D'].set_title(f'Feature component {i+1}')
    axs['D'].set_axis_off()

    plt.show()

In [ ]:
# now do a raster plot just showing the dynamics of each day side by side

## 1 · PSTH heatmaps (representation-drift style)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1 · PSTH heatmaps – representation-drift style
#   Rows = neurons, columns = time-points, one panel per day.
#   Two versions:
#     (a) default neuron ordering (as loaded)
#     (b) neurons sorted by mean response during the stimulus window
#
# Trial layout: pre-stim = frames 0–14  (15 frames)
#               stimulus = frames 15–74 (60 frames)
#               post-stim = last 15 frames
# ─────────────────────────────────────────────────────────────────

STIM_START  = 15          # first stimulus frame (0-indexed)
STIM_FRAMES = 60          # stimulus duration in frames
STIM_END    = STIM_START + STIM_FRAMES   # = 75  (exclusive)
CMAP_PSTH   = 'RdBu_r'   # diverging colormap centred at 0
FIG_DPI     = 200


def psth_heatmap(all_resp_mean, neuron_order, title_prefix,
                 stim_start=STIM_START, stim_end=STIM_END):
    """Plot PSTH heatmaps for every day, preserving a fixed neuron ordering.

    Parameters
    ----------
    all_resp_mean : ndarray, shape (n_days, n_neurons, n_features)
        Trial-averaged responses.
    neuron_order : 1-D int array of length n_neurons
        Row ordering applied to every day panel.
    title_prefix : str
        String prepended to each day's subplot title.
    stim_start : int
        Frame index of stimulus onset.
    stim_end : int
        Frame index of stimulus offset (exclusive).
    """
    n_days, n_neurons, n_features = all_resp_mean.shape

    # Shared colour scale: ±2 SD across all days and neurons
    flat = all_resp_mean[:, neuron_order, :]
    vmax = 2 * np.std(flat)
    vmin = -vmax

    fig, axs = plt.subplots(
        1, n_days,
        figsize=(2.8 * n_days, max(3, n_neurons / 60)),
        dpi=FIG_DPI,
        constrained_layout=True,
    )
    if n_days == 1:
        axs = [axs]

    for d, ax in enumerate(axs):
        img = all_resp_mean[d, neuron_order, :]          # (n_neurons, n_features)
        im  = ax.imshow(
            img,
            aspect='auto',
            cmap=CMAP_PSTH,
            vmin=vmin,
            vmax=vmax,
            interpolation='none',
        )
        # mark stimulus onset and offset
        ax.axvline(stim_start - 0.5, color='k', lw=0.8, ls='--', alpha=0.7)
        ax.axvline(stim_end  - 0.5, color='k', lw=0.8, ls='--', alpha=0.7)
        ax.set_title(f'P{8 + d}', fontsize=10)
        ax.set_xlabel('Time (frames)', fontsize=7)
        if d == 0:
            ax.set_ylabel('Neuron', fontsize=7)
        ax.tick_params(labelsize=6)

    # shared colourbar
    cb = fig.colorbar(im, ax=axs[-1], shrink=0.6, pad=0.02)
    cb.set_label('ΔF/F (z-score)', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    fig.suptitle(title_prefix, fontsize=12, y=1.01)
    plt.show()


# (a) default ordering
default_order = np.arange(n_neurons)
psth_heatmap(all_resp_mean, default_order, 'PSTH – default neuron order')

# (b) sort by mean response during the stimulus window
#     Use day 0 as reference so the order is fixed across all panels
stim_mean_day0 = all_resp_mean[0, :, STIM_START:STIM_END].mean(axis=1)
sorted_order   = np.argsort(stim_mean_day0)[::-1]           # largest response first
psth_heatmap(all_resp_mean, sorted_order,
             'PSTH – neurons sorted by stimulus response (day 1)')


## 2 · Manual-feature correlations

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2 · Manual feature definitions & correlation with neural responses
#
# Trial layout: pre-stim = frames 0–14  (15 frames)
#               stimulus = frames 15–74 (60 frames)
#               post-stim = last 15 frames
# ─────────────────────────────────────────────────────────────────

STIM_START   = 15
STIM_FRAMES  = 60
STIM_END     = STIM_START + STIM_FRAMES   # = 75
AGE_OFFSET   = 8          # P8 on day 0


# ── 2a · Define features ─────────────────────────────────────────

def make_manual_features(n_features, stim_start=STIM_START, stim_frames=STIM_FRAMES):
    """Return a dict of named 1-D feature templates (length = n_features).

    Features are non-zero only inside the stimulus window (frames
    stim_start : stim_start + stim_frames) and are normalised to unit
    ℓ2-norm so Pearson correlations are on the same scale.

    Features
    --------
    on-off  : square pulse (1 during stim, 0 elsewhere)
    sin osc : 5 full sine cycles during the stimulus, zero mean by construction
    ramp    : linear ramp up over the stimulus window  (0 → 1)
    decay   : linear ramp down over the stimulus window (1 → 0)
    """
    stim_end = stim_start + stim_frames
    t_stim   = np.arange(stim_frames, dtype=float)

    def _embed(kernel):
        """Place a stim-length kernel at the correct position in the full vector."""
        f = np.zeros(n_features)
        f[stim_start:stim_end] = kernel
        return f

    on_off  = _embed(np.ones(stim_frames))

    # 5 full periods within the stimulus window; sin has zero mean over full cycles
    sin_osc = _embed(np.sin(2 * np.pi * 5 * t_stim / stim_frames))

    # ramp    = _embed(np.linspace(0, 1, stim_frames))

    # # linear decay (ramp down): peaks at onset, reaches 0 at offset
    # decay   = _embed(np.linspace(1, 0, stim_frames))
    # modify ramp and decay to have mean 0 over the stimulus window (so they are uncorrelated with on-off)
    ramp    = _embed(np.linspace(-0.5, 0.5, stim_frames))
    decay   = _embed(np.linspace(0.5, -0.5, stim_frames))

    features = {'on-off': on_off, 'sin osc': sin_osc, 'ramp': ramp, 'decay': decay}

    # normalise to unit L2 norm
    features = {k: v / (np.linalg.norm(v) + 1e-12) for k, v in features.items()}
    return features


features = make_manual_features(n_features)


# ── 2b · Plot feature templates ──────────────────────────────────

fig, axs = plt.subplots(1, len(features), figsize=(3 * len(features), 2.5), dpi=FIG_DPI)
feature_colors = ['C0', 'C1', 'C2', 'C3']

for ax, (name, f), color in zip(axs, features.items(), feature_colors):
    ax.plot(f, color=color, lw=2)
    ax.axvspan(STIM_START, STIM_END, color='lightgray', alpha=0.3, label='stimulus')
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Time (frames)', fontsize=8)
    ax.set_ylabel('Amplitude', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Manual feature templates', fontsize=12)
plt.tight_layout()
plt.show()


# ── 2c · Compute Pearson correlation for each neuron × day ───────

def compute_feature_correlations(all_resp_mean, features):
    """Compute Pearson r between each neuron-day response and each feature.

    Parameters
    ----------
    all_resp_mean : ndarray (n_days, n_neurons, n_features)
    features      : dict {name: 1-D array of length n_features}

    Returns
    -------
    corr_dict  : dict {name: ndarray shape (n_days * n_neurons,)}
    age_labels : 1-D int array (postnatal day for every data point)
    """
    n_days, n_neurons, nf = all_resp_mean.shape
    X = all_resp_mean.reshape(-1, nf)

    # z-score each row (guard against zero-SD rows)
    X_z   = X - X.mean(axis=1, keepdims=True)
    row_sd = X_z.std(axis=1, keepdims=True)
    row_sd[row_sd == 0] = 1.0
    X_z /= row_sd

    corr_dict = {}
    for name, f in features.items():
        f_z = (f - f.mean()) / (f.std() + 1e-12)
        corr_dict[name] = (X_z * f_z).mean(axis=1)   # Pearson r

    age_labels = np.repeat(np.arange(n_days) + AGE_OFFSET, n_neurons)
    return corr_dict, age_labels


corr_dict, age_labels = compute_feature_correlations(all_resp_mean, features)


# ── 2d · Scatter matrix: each feature vs every other, colour = age ──

feat_names = list(corr_dict.keys())
n_feats    = len(feat_names)

fig, axs = plt.subplots(n_feats, n_feats, figsize=(3 * n_feats, 3 * n_feats), dpi=FIG_DPI)

for r in range(n_feats):
    for c in range(n_feats):
        ax = axs[r, c]
        if r == c:
            # diagonal: per-day histograms
            for d in range(n_days):
                idx   = np.where(age_labels == d + AGE_OFFSET)[0]
                color = plt.cm.plasma(d / (n_days - 1))
                ax.hist(corr_dict[feat_names[r]][idx], bins=20, color=color,
                        alpha=0.5, density=True, histtype='stepfilled')
            ax.set_xlabel(feat_names[r], fontsize=8)
        else:
            ax.scatter(
                corr_dict[feat_names[c]],
                corr_dict[feat_names[r]],
                c=age_labels, cmap='plasma',
                s=2, alpha=0.5, rasterized=True,
            )
            ax.set_xlabel(feat_names[c], fontsize=7)
            ax.set_ylabel(feat_names[r], fontsize=7)

        ax.axhline(0, lw=0.4, color='k', ls='--')
        ax.axvline(0, lw=0.4, color='k', ls='--')
        ax.tick_params(labelsize=6)
        ax.spines[['top', 'right']].set_visible(False)

sm = plt.cm.ScalarMappable(cmap='plasma',
                           norm=plt.Normalize(vmin=AGE_OFFSET, vmax=AGE_OFFSET + n_days - 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs, shrink=0.4, pad=0.02)
cbar.set_label('Postnatal day', fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle('Feature-correlation scatter matrix (colour = postnatal day)', fontsize=12)
plt.show()


## 3 · NMF population dynamics

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3 · NMF population dynamics
#
# The PSTH matrix X (n_neurons × n_features) for each day is
# factorised as  X ≈ W H  where
#   W (n_neurons  × n_components) – neuron loadings
#   H (n_components × n_features) – temporal basis functions
#
# The *columns* of H  (shape n_features × n_components) represent
# each time-point as a point in component space.  We embed those
# n_features time-points into 2-D UMAP to visualise the population
# trajectory through time.  Each day therefore contributes
# n_features (= 90) points to the embedding.
# ─────────────────────────────────────────────────────────────────

from sklearn.decomposition import NMF
import umap

NMF_COMPONENTS = 6     # rank of factorisation
NMF_SEED       = 42


# ── 3a · Helper functions ────────────────────────────────────────

def fit_nmf(X, n_components=NMF_COMPONENTS, seed=NMF_SEED):
    """Fit NMF to a non-negative PSTH matrix.

    Applies ReLU to guarantee non-negativity before fitting.

    Parameters
    ----------
    X : ndarray (n_neurons, n_features)
    n_components : int
    seed : int

    Returns
    -------
    W     : ndarray (n_neurons,   n_components)
    H     : ndarray (n_components, n_features)
    model : fitted sklearn NMF object
    """
    X_nn  = np.clip(X, 0, None)
    model = NMF(n_components=n_components, init='nndsvda',
                random_state=seed, max_iter=500)
    W = model.fit_transform(X_nn)
    H = model.components_
    return W, H, model


def project_nmf(X, model):
    """Project new data onto a pre-fitted NMF basis (returns W only).

    Parameters
    ----------
    X     : ndarray (n_neurons, n_features)
    model : fitted sklearn NMF object

    Returns
    -------
    W_proj : ndarray (n_neurons, n_components)
    """
    return model.transform(np.clip(X, 0, None))


def get_time_coords(H):
    """Return time-point coordinates in component space.

    Parameters
    ----------
    H : ndarray (n_components, n_features)
        NMF temporal basis.

    Returns
    -------
    T : ndarray (n_features, n_components)
        Each row is one time-point's coordinate vector.
    """
    return H.T     # transpose: rows = time-points


def fit_umap_on_coords(T_list, seed=NMF_SEED):
    """Jointly embed a list of time-coordinate matrices into 2-D UMAP.

    Parameters
    ----------
    T_list : list of ndarray, each (n_features, n_components)

    Returns
    -------
    embeddings : list of ndarray, each (n_features, 2)
    reducer    : fitted umap.UMAP
    """
    T_cat   = np.vstack(T_list)
    reducer = umap.UMAP(n_neighbors=min(15, len(T_cat) - 1),
                        random_state=seed, min_dist=0.3)
    emb_all = reducer.fit_transform(T_cat)
    nf      = T_list[0].shape[0]
    embeddings = [emb_all[d * nf:(d + 1) * nf] for d in range(len(T_list))]
    return embeddings, reducer


def plot_trajectories_per_day(embeddings, ages, title,
                              stim_start=STIM_START, stim_end=STIM_END):
    """Plot each day's time-trajectory in its own axis, colour = time.

    Each point is one time-frame; the trajectory is drawn in order.
    The stimulus window is highlighted in each panel.

    Parameters
    ----------
    embeddings : list of ndarray (n_features, 2), one per day
    ages       : 1-D int array of postnatal-day labels
    title      : str  – figure suptitle
    stim_start : int  – first stimulus frame
    stim_end   : int  – last stimulus frame (exclusive)
    """
    n_days  = len(embeddings)
    n_frames = embeddings[0].shape[0]
    t_idx   = np.arange(n_frames)

    fig, axs = plt.subplots(1, n_days, figsize=(3.2 * n_days, 3.2),
                            dpi=FIG_DPI, constrained_layout=True)
    if n_days == 1:
        axs = [axs]

    for d, (ax, emb, age) in enumerate(zip(axs, embeddings, ages)):
        day_color = plt.cm.plasma(d / (n_days - 1))

        # draw full trajectory, coloured by time
        sc = ax.scatter(emb[:, 0], emb[:, 1],
                        c=t_idx, cmap='viridis', s=10, zorder=3)
        ax.plot(emb[:, 0], emb[:, 1], color='grey', lw=0.6, alpha=0.5, zorder=2)

        # highlight stimulus frames
        ax.scatter(emb[stim_start:stim_end, 0],
                   emb[stim_start:stim_end, 1],
                   s=18, color='tomato', zorder=4, label='stimulus', linewidths=0)

        # mark onset and offset
        ax.scatter(*emb[stim_start], s=60, color='k', marker='^', zorder=5)
        ax.scatter(*emb[stim_end - 1], s=60, color='k', marker='v', zorder=5)

        ax.set_title(f'P{age}', fontsize=10, color=day_color)
        ax.set_xlabel('NMF dim 1', fontsize=7)
        if d == 0:
            ax.set_ylabel('NMF dim 2', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle(title, fontsize=11)
    plt.show()


def plot_trajectories_combined(embeddings, ages, title,
                               stim_start=STIM_START, stim_end=STIM_END):
    """Plot all days' trajectories on one axis, colour-coded by age.

    Parameters
    ----------
    embeddings : list of ndarray (n_features, 2)
    ages       : 1-D int array
    title      : str
    """
    n_days = len(embeddings)
    norm   = plt.Normalize(vmin=ages.min(), vmax=ages.max())
    cmap   = plt.cm.plasma

    fig, ax = plt.subplots(figsize=(7, 5), dpi=FIG_DPI)

    for d, (emb, age) in enumerate(zip(embeddings, ages)):
        color = cmap(norm(age))
        ax.plot(emb[:, 0], emb[:, 1], color=color, lw=1.5, alpha=0.8, zorder=2)
        ax.scatter(emb[:, 0], emb[:, 1], color=color, s=6, zorder=3)
        # mark stim onset with a triangle
        ax.scatter(*emb[stim_start], s=60, color=color,
                   marker='^', zorder=5, edgecolors='k', linewidths=0.5)
        ax.text(emb[stim_start, 0], emb[stim_start, 1],
                f' P{age}', fontsize=7, color=color, va='center')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
    cbar.set_label('Postnatal day', fontsize=10)
    cbar.ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('NMF dim 1', fontsize=9)
    ax.set_ylabel('NMF dim 2', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7)
    plt.tight_layout()
    plt.show()


ages = np.arange(n_days) + AGE_OFFSET


# ── 3b · Fit NMF independently for each day ──────────────────────

print('Fitting NMF independently per day …')
T_per_day, W_per_day, models_per_day = [], [], []
for d in range(n_days):
    W, H, model = fit_nmf(all_resp_mean[d])
    W_per_day.append(W)
    T_per_day.append(get_time_coords(H))   # (n_features, n_components)
    models_per_day.append(model)
    print(f'  Day P{ages[d]}: reconstruction error = {model.reconstruction_err_:.4f}')

# Jointly embed all n_features × n_days time-point coordinates
emb_indep, _ = fit_umap_on_coords(T_per_day)
plot_trajectories_per_day(emb_indep, ages,
                          'NMF per day (independent) – time trajectories in NMF space')


# ── 3c · Fit on reference day, project all others ────────────────

for ref_label, ref_idx in [('first day (P8)', 0), ('last day (P13)', n_days - 1)]:
    print(f'\nFitting NMF on {ref_label}, projecting all others …')
    W_ref, H_ref, model_ref = fit_nmf(all_resp_mean[ref_idx])

    T_proj_list = []
    for d in range(n_days):
        W_proj = project_nmf(all_resp_mean[d], model_ref)
        # Reconstruct temporal coords via least-squares: X ≈ W_proj @ H_proj
        X_nn   = np.clip(all_resp_mean[d], 0, None)
        H_proj = np.linalg.lstsq(W_proj, X_nn, rcond=None)[0]  # (n_components, n_features)
        T_proj_list.append(get_time_coords(H_proj))              # (n_features, n_components)

    emb_proj, _ = fit_umap_on_coords(T_proj_list)
    plot_trajectories_per_day(emb_proj, ages,
                              f'NMF basis from {ref_label} – all days projected')


# ── 3d · Concatenate all days, fit one NMF, embed & colour by age ─

print('\nFitting NMF on concatenated PSTHs (all days) …')
X_cat          = np.vstack([all_resp_mean[d] for d in range(n_days)])
W_cat, _, model_cat = fit_nmf(X_cat)
print(f'  Global reconstruction error = {model_cat.reconstruction_err_:.4f}')

# Reconstruct per-day H from the shared model
T_global_list = []
for d in range(n_days):
    W_d  = W_cat[d * n_neurons:(d + 1) * n_neurons]
    X_d  = np.clip(all_resp_mean[d], 0, None)
    H_d  = np.linalg.lstsq(W_d, X_d, rcond=None)[0]   # (n_components, n_features)
    T_global_list.append(get_time_coords(H_d))           # (n_features, n_components)

emb_global, _ = fit_umap_on_coords(T_global_list)
plot_trajectories_combined(emb_global, ages,
                           'NMF on concatenated PSTHs – trajectories colour-coded by age')
